In [1]:
"""
FILE 1: feature_engineering.py
================================
Reads raw data → applies all feature engineering → saves output CSVs.
Run this FIRST, then run model_training.py
"""

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("FEATURE ENGINEERING PIPELINE")
print("=" * 60)

FEATURE ENGINEERING PIPELINE


In [2]:
# ── LOAD RAW DATA ────────────────────────────────────────────────
customer_df = pd.read_csv("../needed/insurance_dataset.csv")
claims_df   = pd.read_csv("../needed/insurance_claims.csv")
print(f"Loaded: customers {customer_df.shape}  |  claims {claims_df.shape}")

Loaded: customers (510, 9)  |  claims (1000, 40)


In [3]:
# ── STEP 1: CLEAN CUSTOMER DATA ──────────────────────────────────
df = customer_df.copy()

gender_map = {'M': 'Male', 'F': 'Female', 'Male': 'Male',
              'Female': 'Female', 'Other': 'Other'}
df['Gender'] = df['Gender'].map(gender_map).fillna('Unknown')

df['Premium'] = df.groupby('Policy_Type')['Premium'].transform(
    lambda x: x.fillna(x.median()))
df['Customer_Satisfaction'] = df['Customer_Satisfaction'].fillna(
    df['Customer_Satisfaction'].median())

print("Gender cleaned, missing values imputed.")

Gender cleaned, missing values imputed.


In [4]:
df['Date_Joined'] = pd.to_datetime(df['Date_Joined'])
reference_date    = pd.Timestamp('2025-01-01')
df['tenure_years'] = (reference_date - df['Date_Joined']).dt.days / 365.25

df['age_group'] = pd.cut(df['Age'], bins=[0, 35, 55, 100],
                          labels=['Young', 'Mid', 'Senior'])

df['premium_tier'] = pd.qcut(df['Premium'], q=3,
                              labels=['Low', 'Mid', 'High'],
                              duplicates='drop')

df['claims_per_year']         = df['Claim_Count'] / df['tenure_years'].replace(0, 0.1)
df['age_x_claims']            = df['Age'] * df['Claim_Count']
df['premium_x_satisfaction']  = df['Premium'] * df['Customer_Satisfaction']
df['satisfaction_per_tenure'] = df['Customer_Satisfaction'] / df['tenure_years'].replace(0, 0.1)

premium_75th        = df['Premium'].quantile(0.75)
df['is_high_value'] = (df['Premium'] > premium_75th).astype(int)

df['satisfaction_group'] = pd.cut(df['Customer_Satisfaction'],
                                   bins=[0, 3, 7, 10],
                                   labels=['Low', 'Mid', 'High'])

In [5]:
ohe_cols = ['Gender', 'Region', 'age_group', 'premium_tier', 'satisfaction_group']
df_encoded = pd.get_dummies(df, columns=ohe_cols, drop_first=False)

# Drop non-feature columns (keep Policy_Type as target, keep Customer_ID for reference)
df_encoded.drop(columns=['Date_Joined'], inplace=True)

print(f"Customer engineered shape: {df_encoded.shape}")
print(f"Columns: {df_encoded.columns.tolist()}")

Customer engineered shape: (510, 29)
Columns: ['Customer_ID', 'Age', 'Policy_Type', 'Premium', 'Claim_Count', 'Customer_Satisfaction', 'tenure_years', 'claims_per_year', 'age_x_claims', 'premium_x_satisfaction', 'satisfaction_per_tenure', 'is_high_value', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Gender_Unknown', 'Region_East', 'Region_North', 'Region_South', 'Region_West', 'age_group_Young', 'age_group_Mid', 'age_group_Senior', 'premium_tier_Low', 'premium_tier_Mid', 'premium_tier_High', 'satisfaction_group_Low', 'satisfaction_group_Mid', 'satisfaction_group_High']


In [6]:
cl = claims_df.copy()
cl.drop(columns=['_c39'], errors='ignore', inplace=True)
cl.replace('?', np.nan, inplace=True)

severity_map = {'Trivial Damage': 1, 'Minor Damage': 2,
                'Major Damage': 3, 'Total Loss': 4}
cl['severity_score']    = cl['incident_severity'].map(severity_map)
cl['is_fraud']          = (cl['fraud_reported'] == 'Y').astype(int)
cl['is_night_incident'] = cl['incident_hour_of_the_day'].apply(
    lambda h: 1 if pd.notnull(h) and (int(h) >= 22 or int(h) <= 6) else 0)
cl['is_collision']      = cl['incident_type'].apply(
    lambda x: 1 if pd.notnull(x) and 'Collision' in str(x) else 0)
cl['vehicle_age']       = 2015 - cl['auto_year'].fillna(2005)
cl['claim_per_premium'] = cl['total_claim_amount'] / cl['policy_annual_premium'].replace(0, np.nan)
cl['injury_ratio']      = cl['injury_claim']  / cl['total_claim_amount'].replace(0, np.nan)
cl['vehicle_ratio']     = cl['vehicle_claim'] / cl['total_claim_amount'].replace(0, np.nan)
cl['property_ratio']    = cl['property_claim'] / cl['total_claim_amount'].replace(0, np.nan)

# Aggregate to policy level
claims_agg = cl.groupby('policy_number').agg(
    total_claim_amount    = ('total_claim_amount',          'sum'),
    avg_injury_claim      = ('injury_claim',                'mean'),
    avg_property_claim    = ('property_claim',              'mean'),
    avg_vehicle_claim     = ('vehicle_claim',               'mean'),
    fraud_rate            = ('is_fraud',                    'mean'),
    has_fraud_history     = ('is_fraud',                    'max'),
    avg_severity          = ('severity_score',              'mean'),
    max_severity          = ('severity_score',              'max'),
    total_incidents       = ('incident_type',               'count'),
    night_incident_rate   = ('is_night_incident',           'mean'),
    collision_rate        = ('is_collision',                'mean'),
    avg_vehicles_involved = ('number_of_vehicles_involved', 'mean'),
    avg_bodily_injuries   = ('bodily_injuries',             'mean'),
    avg_witnesses         = ('witnesses',                   'mean'),
    annual_premium        = ('policy_annual_premium',       'mean'),
).reset_index()

claims_agg['claim_to_premium_ratio'] = (
    claims_agg['total_claim_amount'] /
    claims_agg['annual_premium'].replace(0, np.nan))
claims_agg['risk_score'] = (
    claims_agg['fraud_rate'] * 2 +
    claims_agg['avg_severity'] +
    claims_agg['night_incident_rate'])

# Encode claims categoricals
cat_cols = ['insured_sex', 'insured_education_level', 'insured_relationship',
            'incident_type', 'collision_type', 'incident_severity',
            'authorities_contacted', 'policy_state', 'policy_csl',
            'insured_occupation', 'property_damage', 'police_report_available']
cl_encoded = pd.get_dummies(cl, columns=[c for c in cat_cols if c in cl.columns],
                             drop_first=True)

print(f"Claims engineered shape  : {cl_encoded.shape}")


Claims engineered shape  : (1000, 78)


In [7]:
# 5a. Customer engineered dataset (for Model 1 & 2)
df_encoded.to_csv("customer_engineered.csv", index=False)
print("\nSaved: customer_engineered.csv")

# 5b. Claims aggregated per policy (for joining if needed)
claims_agg.to_csv("claims_aggregated.csv", index=False)
print("Saved: claims_aggregated.csv")

# 5c. Full claims engineered (for Model 3 - Fraud)
cl_encoded.to_csv("claims_engineered.csv", index=False)
print("Saved: claims_engineered.csv")

# 5d. Save metadata needed by model training
#     (column lists, thresholds, encoders — so model_training.py stays consistent)
metadata = {
    'premium_75th'    : float(premium_75th),
    'reference_date'  : str(reference_date),
    'ohe_cols'        : ohe_cols,
    'severity_map'    : severity_map,
    'customer_shape'  : df_encoded.shape,
    'claims_shape'    : cl_encoded.shape,
    'customer_columns': df_encoded.columns.tolist(),
    'claims_columns'  : cl_encoded.columns.tolist(),
}
with open("engineering_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)
    print("Saved: engineering_metadata.pkl  (metadata for model training)")

print("\n" + "=" * 60)
print("Feature engineering complete.")
print("Next step: run  python model_training.py")
print("=" * 60)


Saved: customer_engineered.csv
Saved: claims_aggregated.csv
Saved: claims_engineered.csv
Saved: engineering_metadata.pkl  (metadata for model training)

Feature engineering complete.
Next step: run  python model_training.py
